<a href="https://colab.research.google.com/github/antonypradeep54/RAG-research-paper-qa-system/blob/main/Capstone_Project_Research_Paper_Question_%26_Answer_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font size=+4 color="#009688"><center><b> CAPSTONE PROJECT </b></center></font>

<font size=+3 color="#009688"><center><b> Research Paper Question & Answer Bot</b></center></font>

## Table of contents

- Problem statement
- Libraries and configurations
- Collect Research Paper
    - Research Paper Fetcher
    - Research Paper Metadata Parser
    - Research Paper Downloader
    - Orchestration Research Paper Collection
- Preprocess, Clean, and Chunk the Research Papers
- Convert chunks to LangChain Document
- OpenAI RAG
    - Embedding, Vector Store, and Retrieval Engine
    - Basic RAG Query
    - LCEL RAG Query
- HuggingFace RAG
    - Embedding, Vector Store, and Retrieval Engine
    - Basic RAG Query
    - LCEL RAG Query
- Semantic Similarity Evaluation
- Conclusion
- References

##Problem Statement

The scenario is that you are working for a company like ArXiv which has a repository of the latest and best papers around AI and Data Science. They want to build an intelligent chatbot which can help answer questions which people ask on popular topics based on research papers in Generative AI. Considering the limited time-frame of the project, we have limited the dataset
to a few documents. </br>

The broad idea of this project is to build a RAG system on top of some of the famous seminal research papers around Generative AI and LLMs, this system should be able to index the papers into a vector database, use good retrieval strategies to retrieve relevant contextual documents based on your input queries and generate proper responses. The focus of this project is not just to build a simple RAG system but explore various approaches and methodologies in each component when building the RAG system and then finally choose the
approach and methodology which works best for you. </br>

You are not compelled to use the sample data we provide for this project, if you have your own documents or data, you are more than welcome to use it (but validate your data and idea with us first), the project steps mentioned below would remain unchanged.

##Libraries and configurations

In [ ]:
import os
import requests
import xml.etree.ElementTree as ET
from urllib.parse import urlencode

# -----------------------------
# CONFIGURATION
# -----------------------------
SEARCH_QUERY = "Generative AI"       # Your keyword
MAX_RESULTS = 20                     # Number of papers to fetch
DOWNLOAD_DIR = "arxiv_papers"       # Folder to save PDFs

# Create download directory if it doesn't exist
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

##Functions

###Research Paper Fetcher

##### The purpose of this function is to send a query to the arXiv public API (https://arxiv.org/help/api) and fetch results matching a given keyword or topic. <\br><\br>

It returns a XML file with metadata. such as Title, Authors, Summary, PDF Link for further parsing.

In [ ]:
# -----------------------------
# FUNCTION TO QUERY ARXIV API
# -----------------------------
def query_arxiv(search_query, start=0, max_results=20):
    base_url = "http://export.arxiv.org/api/query?"
    params = {
        "search_query": f"all:{search_query}",
        "start": start,
        "max_results": max_results,
        "sortBy": "relevance",
        "sortOrder": "descending"
    }
    query_url = base_url + urlencode(params)
    response = requests.get(query_url)
    if response.status_code != 200:
        raise Exception(f"API request failed with status {response.status_code}")
    return response.text

###Research Paper Metadata Parser

##### The function parse_arxiv(xml_data) is designed to parse (i.e., extract structured information) from the XML response returned by the previous function query_arxiv().

In [ ]:
# -----------------------------
# FUNCTION TO PARSE ARXIV RESULTS
# -----------------------------
def parse_arxiv(xml_data):
    root = ET.fromstring(xml_data)
    ns = {'atom': 'http://www.w3.org/2005/Atom'}
    papers = []
    for entry in root.findall('atom:entry', ns):
        title = entry.find('atom:title', ns).text.strip().replace('\n', ' ')
        authors = [a.find('atom:name', ns).text for a in entry.findall('atom:author', ns)]
        summary = entry.find('atom:summary', ns).text.strip().replace('\n', ' ')
        pdf_link = ""
        for link in entry.findall('atom:link', ns):
            if link.attrib.get('title') == 'pdf':
                pdf_link = link.attrib['href']
        papers.append({
            "title": title,
            "authors": authors,
            "summary": summary,
            "pdf_link": pdf_link
        })
    return papers

###Research Paper Downloader

#####Download the actual PDF file of a research paper from arXiv using the direct PDF link obtained from the XML. It also ensures you don’t download the same paper twice (by checking if the file already exists).

In [ ]:
# -----------------------------
# FUNCTION TO DOWNLOAD PDF
# -----------------------------
def download_pdf(pdf_url, save_dir):
    if not pdf_url:
        return None
    filename = pdf_url.split('/')[-1] + ".pdf"
    filepath = os.path.join(save_dir, filename)
    if os.path.exists(filepath):
        print(f"{filename} already exists, skipping...")
        return filepath
    print(f"Downloading {filename} ...")
    response = requests.get(pdf_url)
    with open(filepath, 'wb') as f:
        f.write(response.content)
    return filepath

###Orchestration Research Paper Collection

#####This script automates the complete arXiv research paper collection process. It’s the glue that connects all three functions query_arxiv() ➜ parse_arxiv() ➜ download_pdf()

- Search arXiv for a given topic (SEARCH_QUERY)
- Parse the results into structured data (title, authors, summary, PDF link)
- Download each paper’s PDF into a local folder

In [ ]:
# -----------------------------
# ORCHESTRATION SCRIPT
# -----------------------------
if __name__ == "__main__":
    print(f"Querying arXiv for '{SEARCH_QUERY}' research papers...")
    print('\n')
    xml_data = query_arxiv(SEARCH_QUERY, max_results=MAX_RESULTS)
    papers = parse_arxiv(xml_data)

    print(f"Found {len(papers)} papers. Downloading PDFs ...")
    print('\n')
    for paper in papers:
        print(f"Title: {paper['title']}")
        print(f"Authors: {', '.join(paper['authors'])}")
        print('\n')
        download_pdf(paper['pdf_link'], DOWNLOAD_DIR)

    print("All done!")

Querying arXiv for 'Generative AI' research papers...


Found 20 papers. Downloading PDFs ...


Title: Foundations of GenIR
Authors: Qingyao Ai, Jingtao Zhan, Yiqun Liu


Title: AI and the EU Digital Markets Act: Addressing the Risks of Bigness in Generative AI
Authors: Ayse Gizem Yasar, Andrew Chong, Evan Dong, Thomas Krendl Gilbert, Sarah Hladikova, Roland Maio, Carlos Mougan, Xudong Shen, Shubham Singh, Ana-Andreea Stoica, Savannah Thais, Miri Zilka


Title: Ultra Strong Machine Learning: Teaching Humans Active Learning Strategies via Automated AI Explanations
Authors: Lun Ai, Johannes Langer, Ute Schmid, Stephen Muggleton


Title: Exploring utilization of generative AI for research and education in data-driven materials science
Authors: Takahiro Misawa, Ai Koizumi, Ryo Tamura, Kazuyoshi Yoshimi


Title: Business (mis)Use Cases of Generative AI
Authors: Stephanie Houde, Vera Liao, Jacquelyn Martino, Michael Muller, David Piorkowski, John Richards, Justin Weisz, Yunfeng Zhang


Title

##Preprocess, Clean, and Chunk the Research Papers

#####The following function prepares the data for RAG by extracting, cleaning, chunking, and creating dictionary for each chunk.

In [ ]:
%%capture
!pip install pdfplumber

######<strong>extract_text_from_pdf</strong> : Opens the PDF at pdf_path using pdfplumber. Iterates page by page and extracts text. Concatenates all pages into a single string with \n between pages. Returns the raw text of the PDF.

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract text from a single PDF using pdfplumber
    """
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

#####<strong>clean_text</strong> : Removes LaTeX equations replaced with a space. Removes page numbers patterns like Page 1 or 1/10. Normalizes whitespace converts multiple spaces/newlines into a single space. Returns cleaned text.

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean the extracted text:
    - Remove multiple newlines
    - Remove extra whitespaces
    - Remove simple page numbers
    - Remove LaTeX equations between $...$
    """
    # Remove LaTeX math equations like $...$ or $$...$$
    text = re.sub(r'\$\$.*?\$\$|\$.*?\$', ' ', text, flags=re.DOTALL)

    # Remove page numbers like "Page 1", "1/10", etc.
    text = re.sub(r'Page \d+|\d+/\d+', ' ', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

#####<strong>chunk_text</strong> : Splits text into smaller chunks of roughly max_words words (default 500). This helps embedding models and RAG systems because they work better with smaller text segments. Returns a list of text chunks.

In [ ]:
from typing import List, Dict

def chunk_text(text: str, max_words: int = 500) -> List[str]:
    """
    Split text into chunks of max_words words
    """
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i+max_words])
        chunks.append(chunk)
    return chunks

#####<strong>preprocess_pdfs</strong> : Orchestration extract_text_from_pdf, clean_text, and chunk_text. Creates a dictionary with pdf_file, chunk_id, and text. Returns a list of dictionaries, each representing a chunk.


In [ ]:
def preprocess_pdfs(pdf_dir: str, chunk_size: int = 500) -> List[Dict]:
    """
    Process all PDFs in a folder and return a list of chunks with metadata
    """
    processed_data = []
    pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith('.pdf')]

    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_dir, pdf_file)
        print(f"Processing: {pdf_file}")

        text = extract_text_from_pdf(pdf_path)
        cleaned_text = clean_text(text)
        chunks = chunk_text(cleaned_text, max_words=chunk_size)

        for i, chunk in enumerate(chunks):
            processed_data.append({
                "pdf_file": pdf_file,
                "chunk_id": i,
                "text": chunk
            })

    return processed_data

In [ ]:
%%capture
!pip install -q PyPDF2

In [ ]:
import os
import tiktoken
from typing import List, Dict
from PyPDF2 import PdfReader

def preprocess_pdfs_token_aware(pdf_dir: str, chunk_token_limit: int = 1000) -> List[Dict]:
    """
    Process all PDFs in a folder and return a list of token-limited text chunks with metadata.
    Each chunk will not exceed the specified token limit.
    """
    processed_data = []
    pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith('.pdf')]
    encoding = tiktoken.get_encoding("cl100k_base")

    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_dir, pdf_file)
        reader = PdfReader(pdf_path)
        metadata_title = reader.metadata.title if reader.metadata and reader.metadata.title else pdf_file

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            tokens = encoding.encode(text)

            # Split into chunks
            chunk_id = 0
            for i in range(0, len(tokens), chunk_token_limit):
                chunk_tokens = tokens[i:i + chunk_token_limit]
                chunk_text = encoding.decode(chunk_tokens)

                processed_data.append({
                    "pdf_file": pdf_file,
                    "page_number": page_number,
                    "pdf_title": metadata_title,
                    "chunk_id": f"{pdf_file}_p{page_number}_c{chunk_id}",
                    "text": chunk_text
                })

                chunk_id += 1

    return processed_data


In [ ]:
# Set your PDF folder path
pdf_folder = "/content/arxiv_papers"   # change this to your folder path

# Call your function
chunks = preprocess_pdfs_token_aware(pdf_folder, chunk_token_limit=1000)

# Make sure you actually got some chunks
print("Total chunks generated:", len(chunks))


Total chunks generated: 354


In [ ]:
# Display first sample
if len(chunks) > 0:
    sample = chunks[100]
    print(
        f"PDF File: {sample.get('pdf_file', 'Unknown')}, \n\n"
        f"Page Number: {sample.get('page_number', 'Unknown')}\n\n"
        f"PDF Title: {sample.get('pdf_title', 'Unknown')}\n\n"
        f"Sample chunk:\n{sample.get('text','')[:500]} ..."
    )
else:
    print("No chunks generated. Check folder path or PDF content.")


PDF File: 2112.01298v2.pdf, 

Page Number: 28

PDF Title: 2112.01298v2.pdf

Sample chunk:
28 Meaningful human control: actionable properties for AI system development
can help understand how, for example, humans may ooad value-laden behav-
ior onto the technology around us. In the \sociology of a door closer", [99]
describes how we made door closers the element in the assembly that mani-
fests politeness by ensuring the door closes softly and gradually, even as the
human actors may barge through without any action to regulate the door).
This sort of division of moral-labor should no ...


In [ ]:
# Number of consecutive chunks to display
N = 5
start_index = 30  # starting chunk index

if len(chunks) > start_index:
    # Extract 5 consecutive chunks
    selected = chunks[start_index : start_index + N]

    # Print PDF title only once
    pdf_title = selected[0].get("pdf_title", "Unknown")
    pdf_file = selected[0].get("pdf_file", "Unknown")

    print(f"PDF File: {pdf_file}")
    print(f"PDF Title: {pdf_title}\n")

    # Collect distinct page numbers
    page_numbers = sorted({c.get("page_number", "Unknown") for c in selected})
    print(f"Page Numbers in these chunks: {page_numbers}\n")

    # Print chunk details (ID + first 500 characters)
    for i, chunk in enumerate(selected, start=start_index):
        print(f"--- Chunk ID: {i} ---")
        print(f"Page Number: {chunk.get('page_number', 'Unknown')}")
        text = chunk.get("text", "")
        print(f"Text (first 500 chars):\n{text[:500]} ...\n")

else:
    print("Not enough chunks generated. Check folder path or PDF content.")


PDF File: 2504.16770v1.pdf
PDF Title: DeBiasMe: De-biasing Human-AI Interactions with Metacognitive AIED (AI in Education) Interventions

Page Numbers in these chunks: [8, 9, 10, 11]

--- Chunk ID: 30 ---
Page Number: 8
Text (first 500 chars):
8 Chaeyeon Lim
limitations in perspectives. This contextual adaptation would enhance the tool’s relevance across various academic
activities and learners.
Fig. 4. Developing bias visualization: A structured Comparison of Biased and Debiased Alternatives
5.3 Bias Analysis and Mitigation Approaches
Developing quantitative metrics for bias detection and mitigation of the tool requires further technical considerations
to develop a multi-dimensional bias index framework for bias detection and mitigat ...

--- Chunk ID: 31 ---
Page Number: 9
Text (first 500 chars):
DeBiasMe: De-biasing Human-AI Interactions with Metacognitive AIED (AI in Education) Interventions 9
6 Discussion
This position paper contributes to the emerging field of AI-augmented reason

##Convert chunks to LangChain Document

In [ ]:
%%capture
!pip install pdfplumber
import pdfplumber

In [ ]:
%%capture
!pip install langchain --upgrade
!pip uninstall -y langchain langchain-core
!pip install langchain

In [ ]:
import langchain
print(langchain.__version__)


1.0.5


In [ ]:
from langchain_core.documents import Document

def convert_to_LangChain_documents(chunks):
    documents = []
    for chunk in chunks:
        documents.append(
            Document(
                page_content=chunk["text"],
                metadata={
                    "pdf_file": chunk["pdf_file"],
                    "page_number": chunk["page_number"],
                    "pdf_title": chunk["pdf_title"],
                    "chunk_id": chunk["chunk_id"],
                },
            )
        )
    return documents


In [ ]:
LangChain_documents = convert_to_LangChain_documents(chunks)

In [ ]:
print(f"Total chunks: {len(LangChain_documents)}")

Total chunks: 354


In [ ]:
import random
from IPython.display import Markdown, display

# Generate a random index (adjust upper bound to match your list length)
random_num = random.randint(1, len(LangChain_documents) - 1)

# Get a random document chunk
doc = LangChain_documents[random_num]

# Safely extract text content
if hasattr(doc, "page_content"):
    sample_text = doc.page_content
elif hasattr(doc, "text"):
    sample_text = doc.text
else:
    raise AttributeError("Unknown document format — no 'text' or 'page_content' attribute found.")

# Extract metadata (if available)
metadata = getattr(doc, "metadata", {})
source = metadata.get("source", "Unknown Source")
page = metadata.get("page", "N/A")

# Take the first 500 characters (optional)
sample_text = sample_text[:500]

# Replace newlines with Markdown line breaks for better readability
sample_text_md = sample_text.replace('\n', '  \n')

# Display as Markdown with metadata header
display(Markdown(f"""
### 📄 Sample Document Chunk
**Source:** {source}
**Page:** {page}
**Chunk Index:** {random_num}

---

{sample_text_md}
"""))



### 📄 Sample Document Chunk
**Source:** Unknown Source
**Page:** N/A
**Chunk Index:** 305

---

 the relevant data is obtained, a suitable model architec-  
ture must be conceived and developed. Since deep learning  
remains more an engineering discipline rather than actual  
science with strong theoretical foundations, this requires  
access to a large number of highly skilled engineers. The  
salaries for these engineers are beyond imagination for many  
individuals, and usually go into six-digit territory, if not be-  
yond. Only some of the most well-funded companies, like  
those in tech, are able to


In [ ]:
%%capture
!pip install --upgrade langchain

##OpenAI RAG


###Embedding, Vector Store, and Retrieval Engine

#####The purpose of this code is to generate vector embeddings for text chunks using OpenAI’s cloud API. OpenAI code runs remotely, calling the API for each text chunk. OpenAI may cost money per request, whereas local models are free after download.

In [ ]:
import os
from google.colab import userdata
# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = userdata.get('OpenAI_API_Key')

In [ ]:
%%capture
!pip install -q langchain langchain-community langchain-openai faiss-cpu tiktoken

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from typing import List
from tqdm import tqdm


class OpenAIEmbeddingComponent:
    """
    LCEL-compatible component for OpenAI embeddings + FAISS.
    """

    def __init__(self, model_name: str = "text-embedding-3-small"):
        self.embedding_model = OpenAIEmbeddings(model=model_name)
        self.vectorstore = None

    def create_vectorstore(self, docs: List[Document], batch_size: int = 50, save_path: str = "faiss_index"):
        """
        Create FAISS vector store from LangChain documents in batches.
        """
        total_docs = len(docs)
        print(f"Total documents to embed: {total_docs}")

        for i in tqdm(range(0, total_docs, batch_size), desc="Embedding documents in batches"):
            batch = docs[i:i + batch_size]
            if self.vectorstore is None:
                self.vectorstore = FAISS.from_documents(batch, self.embedding_model)
            else:
                batch_store = FAISS.from_documents(batch, self.embedding_model)
                self.vectorstore.merge_from(batch_store)

        # Save FAISS index locally
        self.vectorstore.save_local(save_path)
        print(f"✅ Embeddings created and stored at '{save_path}'")

    def load_vectorstore(self, save_path: str = "faiss_index"):
        """
        Load FAISS index from disk (safe if you trust the file source).
        """
        self.vectorstore = FAISS.load_local(
            save_path,
            self.embedding_model,
            allow_dangerous_deserialization=True
        )
        print(f"✅ Vectorstore loaded from '{save_path}'")

    def get_retriever(self, k: int = 4):
        """
        Returns a retriever for use in RAG or LCEL chains.
        """
        if self.vectorstore is None:
            raise ValueError("Vectorstore not initialized. Run create_vectorstore or load_vectorstore first.")
        return self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": k}
        )


In [ ]:
# Initialize component
openai_component = OpenAIEmbeddingComponent()

# 1️⃣ Create FAISS vector store from your document chunks
openai_component.create_vectorstore(LangChain_documents, batch_size=50, save_path="faiss_index")

# 2️⃣ Or load an existing FAISS index
# openai_component.load_vectorstore("faiss_index")

# 3️⃣ Get retriever for RAG
retriever = openai_component.get_retriever(k=4)


Total documents to embed: 354


Embedding documents in batches: 100%|██████████| 8/8 [00:11<00:00,  1.47s/it]

✅ Embeddings created and stored at 'faiss_index'


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableMap, RunnablePassthrough
from langchain_core.prompts import PromptTemplate

In [ ]:
# Initialize your embedding component (already created in your code)
openai_component = OpenAIEmbeddingComponent()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
save_dir = "/content/drive/MyDrive/my_faiss_index"

openai_component.create_vectorstore(
    LangChain_documents,
    batch_size=50,
    save_path=save_dir
)

Total documents to embed: 354


Embedding documents in batches: 100%|██████████| 8/8 [00:10<00:00,  1.26s/it]


✅ Embeddings created and stored at '/content/drive/MyDrive/my_faiss_index'


In [ ]:
openai_component.load_vectorstore(save_dir)

✅ Vectorstore loaded from '/content/drive/MyDrive/my_faiss_index'


In [ ]:
# Get retriever
retriever = openai_component.get_retriever(k=4)


###Basic RAG Query

In [ ]:
from langchain_openai import ChatOpenAI
from IPython.display import Markdown, display

model = ChatOpenAI(model="gpt-4o-mini")

query = "Tell me about BERT4beam"
docs = retriever.invoke(query)
context = "\n\n".join([d.page_content for d in docs])

prompt = f"""
Give a very short summary (3–4 sentences) using the context only.

Context:
{context}

Short Answer:
"""

response = model.invoke(prompt)

# Create a title dynamically from the query
title = f"Short Summary of: {query}"

# Optional: add line breaks between sentences
summary = response.content.replace(". ", ".  \n")

# Display nicely formatted Markdown
display(Markdown(f"### {title}\n\n{summary}"))


### Short Summary of: Tell me about BERT4beam

The paper presents BERT4beam, a novel BERT-based framework designed for optimizing beamforming in wireless networks by treating beamforming tasks as a token-level sequence learning problem.  
It involves a single-task approach utilizing CSI tokenization and a tailored BERT model for generating beamforming vectors, along with pre-training and fine-tuning strategies.  
Additionally, a multi-task extension called UBERT is introduced, which employs element-wise tokenization and incorporates an antenna encoding block to enhance adaptability across various beamforming tasks.  
The proposed models demonstrate strong performance and robustness in dynamic wireless environments through extensive numerical experiments.

###LCEL RAG Query

In [ ]:
from langchain_core.runnables import RunnableLambda

def combine_docs(docs):
    return "\n\n".join([d.page_content for d in docs])


In [ ]:
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an expert assistant. Use ONLY the information from the context to answer.
If the answer is not found in the context, say "The document does not contain this information."

Context:
{context}

Question:
{question}

Answer:
"""
)


In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")


In [ ]:
query = "Tell me about BERT4beam?"



In [ ]:
rewrite_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
Rewrite the question so that it matches the terminology used in the documents.
Only rewrite — do not answer.

Original: {question}
Rewritten:
"""
)

query_rewriter = rewrite_prompt | llm | (lambda x: x.content.strip())


In [ ]:
rag_chain = (
    RunnableMap({
        "context": query_rewriter | retriever | RunnableLambda(combine_docs),
        "question": RunnablePassthrough()
    })
    | prompt_template
    | llm
)


In [ ]:
from IPython.display import display, HTML

# Your query
query = "Tell me about BERT4beam in short format"

# Invoke RAG chain
response = rag_chain.invoke(query)

# Optional: add line breaks for readability
output_text = response.content.replace(". ", ".  \n")

# Example: callback object for demonstration
cb = type("CallbackMock", (), {})()
cb.prompt_tokens = 50
cb.completion_tokens = 120
cb.total_tokens = cb.prompt_tokens + cb.completion_tokens
cb.total_cost = 0.0008

# Optional: some sample markdown
sample_text_md = "<em>Additional notes or sample text can go here.</em>"

# Display nicely in HTML (cleaner look, transparent background)
display(HTML(f"""
<div style="
    font-family: Arial, sans-serif;
    font-size: 14px;
    border: 1px solid #ccc;
    padding: 10px;
    border-radius: 8px;
    background-color: transparent;  /* changed from #f9f9f9 */
">
    📄 <strong>Result Summary</strong><br><br>

    <strong>Query:</strong><br>
    <div style="margin-left: 20px; margin-bottom: 10px;">{query}</div>

    <strong>Result:</strong><br>
    <div style="margin-left: 20px; margin-bottom: 10px;">{output_text}</div>

    <strong>Tokens & Cost:</strong><br>
    <div style="margin-left: 20px;">
        Prompt tokens: {cb.prompt_tokens}<br>
        Completion tokens: {cb.completion_tokens}<br>
        Total tokens: {cb.total_tokens}<br>
        Estimated cost (USD): ${cb.total_cost:.6f}
    </div>

    <hr>
    {sample_text_md}
</div>
"""))


###HuggingFace RAG

###Embedding, Vector Store, and Retrieval Engine

#####The purpose of this code is to generate vector embeddings for text chunks using a Hugging Face SentenceTransformer model. This code does not use the Hugging Face API. Instead, it works locally using a pre-trained model downloaded from Hugging Face’s model hub. Here’s the distinction

In [ ]:
%%capture
!pip install sentence-transformers faiss-cpu

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from typing import List
from tqdm import tqdm


class HuggingFaceEmbeddingComponent:
    """
    LCEL-compatible component for local HuggingFace embeddings + FAISS.
    No API calls. Uses a small, CPU-friendly model.
    """

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        # Load local model
        self.embedding_model = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={"device": "cpu"}  # Force CPU for universal use
        )
        self.vectorstore = None

    def create_vectorstore(self, docs: List[Document], batch_size: int = 50, save_path: str = "faiss_hf_index"):
        """
        Create FAISS vector store from LangChain documents in batches.
        """
        total_docs = len(docs)
        print(f"Total documents to embed: {total_docs}")

        for i in tqdm(range(0, total_docs, batch_size), desc="Embedding documents in batches"):
            batch = docs[i:i + batch_size]

            # First batch creates the FAISS store
            if self.vectorstore is None:
                self.vectorstore = FAISS.from_documents(batch, self.embedding_model)
            else:
                batch_store = FAISS.from_documents(batch, self.embedding_model)
                self.vectorstore.merge_from(batch_store)

        # Save FAISS index locally
        self.vectorstore.save_local(save_path)
        print(f"✅ HF embeddings created and stored at '{save_path}'")

    def load_vectorstore(self, save_path: str = "faiss_hf_index"):
        """
        Load FAISS index from disk (local only).
        """
        self.vectorstore = FAISS.load_local(
            save_path,
            self.embedding_model,
            allow_dangerous_deserialization=True
        )
        print(f"✅ HF Vectorstore loaded from '{save_path}'")

    def get_retriever(self, k: int = 4):
        """
        Returns a retriever using the local HF embedding model.
        """
        if self.vectorstore is None:
            raise ValueError("Vectorstore not initialized. Run create_vectorstore or load_vectorstore first.")

        return self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": k}
        )


In [ ]:
# Initialize your embedding component (already created in your code)
%%capture
huggingface_component = HuggingFaceEmbeddingComponent()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
save_dir_hf = "/content/drive/MyDrive/hf_my_faiss_index"

In [ ]:
huggingface_component.create_vectorstore(
    LangChain_documents,
    batch_size=50,
    save_path=save_dir_hf
)

Total documents to embed: 354


Embedding documents in batches: 100%|██████████| 8/8 [01:24<00:00, 10.55s/it]


✅ HF embeddings created and stored at '/content/drive/MyDrive/hf_my_faiss_index'


In [ ]:
huggingface_component.load_vectorstore(save_dir_hf)

✅ HF Vectorstore loaded from '/content/drive/MyDrive/hf_my_faiss_index'


In [ ]:
# Get retriever
hf_retriever = huggingface_component.get_retriever(k=4)

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

In [ ]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

pipe = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1  # CPU
)

hf_model = HuggingFacePipeline(pipeline=pipe)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu
/tmp/ipython-input-2087560918.py:13: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  hf_model = HuggingFacePipeline(pipeline=pipe)


###Basic RAG Query

In [ ]:
query = "Tell me about BERT4beam"
docs = hf_retriever.invoke(query)
context = "\n\n".join([d.page_content for d in docs])

prompt = f"""
Give a very short summary (3–4 sentences) using the context only.

Context:
{context}

Short Answer:
"""

hf_response = hf_model.invoke(prompt)

hf_title = f"Short Summary of: {query}"
hf_summary = hf_response.replace(". ", ".  \n")

display(Markdown(f"### {hf_title}\n\n{hf_summary}"))


Token indices sequence length is longer than the specified maximum sequence length for this model (3777 > 512). Running this sequence through the model will result in indexing errors


### Short Summary of: Tell me about BERT4beam

A novel framework based on bidirectional encoder representations from transformers (BERT), termed BERT4beam.  
We propose a novel framework based on bidirectional encoder representations from transformers (BERT), termed BERT4beam.  
We propose a novel framework based on bidirectional encoder representations from transformers (BERT), termed BERT4beam.  
We aim to formulate the beamforming optimization problem as a token-level sequence learning task, perform tokenization of the channel state information, construct the BERT model, and con- duct task-specific pre-training and fine-tuning strategies.  
Based on the framework, we propose two BERT-based approaches for single-task and multi-task beamforming optimization, respec- tively.  
Both approaches are generalizable for varying user scales.  
Moreover, the former can adapt to varying system utilities and antenna configurations by re-configuring the input and output module of the BERT model, while the latter, termed UBERT, can directly generalize to diverse tasks, due to a finer-grained tokenization strategy.

###LCEL RAG Query

In [ ]:
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an expert assistant. Use ONLY the information from the context to answer.
If the answer is not found in the context, say "The document does not contain this information."

Context:
{context}

Question:
{question}

Answer:
"""
)

In [ ]:
from langchain_core.runnables import RunnableMap, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
hf_lcel_rag_chain = (
    # 1 - Prepare inputs
    RunnableMap({
        #"context": lambda x: "\n\n".join([d.page_content for d in hf_retriever.get_relevant_documents(x["question"])]),
        "context": lambda x: retriever.invoke(x["question"]),
        "question": RunnablePassthrough()
    })
    # 2 - Feed to prompt template
    | prompt_template
    # 3 - Run FLAN-T5
    | hf_model
    # 4 - Parse text
    | StrOutputParser()
)

In [ ]:
query = "Tell me about BERT4beam in short format"

hf_response = hf_lcel_rag_chain.invoke({"question": query})

# --- If you want manual token counting (optional) ---
# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("your-model-name")
# prompt_tokens = len(tokenizer.encode(query))
# completion_tokens = len(tokenizer.encode(response))
# total_tokens = prompt_tokens + completion_tokens

hf_prompt_tokens = 0            # HF offline models don't return token counts
hf_completion_tokens = 0        # Set manually if needed
hf_total_tokens = 0
hf_estimated_cost = 0.0         # Local HF inference has no cost

# ---- Printing ----
print(f"Query:\n{query}\n")
print(f"Result:\n{hf_response}\n")
print("Tokens & Cost:")
print(f"Prompt tokens: {hf_prompt_tokens}")
print(f"Completion tokens: {hf_completion_tokens}")
print(f"Total tokens: {hf_total_tokens}")
print(f"Estimated cost (USD): ${hf_estimated_cost:.6f}")


Query:
Tell me about BERT4beam in short format

Result:
BERT4beam: Large AI Model Enabled Generalized Beamforming Optimization, 'chunk_id': '2509.11056v1.pdf', 'page_number': 2, 'pdf_title': 'BERT4beam: Large AI Model Enabled Generalized Beamforming Optimization', 'chunk_id': '2509.11056v1.pdf', 'page_number': 2, 'pdf_title': 'BERT4beam: Large AI Model Enabled Generalized Beamforming Optimization', 'chunk_id': '2509.11056v1.pdf', 'page_number': 2, 'pdf_title': 'BERT4beam: Large AI Model Enabled Generalized Beamforming Optimization', 'chunk_id': '2509.11056v1.pdf', 'page_number': 2, 'pdf_title': 'BERT4beam: Large AI Model Enabled Generalized Beamforming Optimization', 'chunk_

Tokens & Cost:
Prompt tokens: 0
Completion tokens: 0
Total tokens: 0
Estimated cost (USD): $0.000000


In [ ]:
from IPython.display import HTML, display

# sample_text_md can contain HTML or markdown-like content
sample_text_md = "<em>Additional notes or sample text can go here.</em>"

display(HTML(f"""
<div style="
    font-family: Arial, sans-serif;
    font-size: 14px;
    border: 1px solid #ccc;
    padding: 15px;
    border-radius: 10px;
    background-color: transparent;  /* changed from #f9f9f9 */
">
    <h3 style="margin-top: 0;">📄 <strong>Result Summary</strong></h3>

    <strong>Query:</strong>
    <div style="margin-left: 20px; margin-bottom: 15px;">
        {query}
    </div>

    <strong>Result:</strong>
    <div style="margin-left: 20px; margin-bottom: 15px; white-space: pre-wrap;">
        {hf_response}
    </div>

    <strong>Tokens & Cost:</strong>
    <div style="margin-left: 20px; margin-bottom: 15px;">
        Prompt tokens: {hf_prompt_tokens}<br>
        Completion tokens: {hf_completion_tokens}<br>
        Total tokens: {hf_total_tokens}<br>
        Estimated cost (USD): ${hf_estimated_cost:.6f}
    </div>

    <hr style="border-top: 1px solid #ddd;">

    <div style="margin-top: 10px;">
        {sample_text_md}
    </div>
</div>
"""))


##Semantic Similarity Evaluation and Comparison

What is Semantic Similarity?

Semantic similarity measures how close two pieces of text are in meaning, even if they don’t share the same words.

For example:

"BERT4Beam is a model for beamforming."

"BERT4Beam is a transformer used in beamforming tasks." </br></br>
These are semantically similar even though the words differ slightly. </br></br>

We compute semantic similarity using vector embeddings. Convert each text into a high-dimensional vector that captures its meaning. Compare vectors using cosine similarity. Cosine similarity ranges from 0 to 1 </br>


1 → texts are very similar

0 → texts are unrelated

In [ ]:
from langchain_openai import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Reference answer
reference = "BERT4Beam is a transformer model for beamforming..."

# OpenAI RAG output
OpenAI_output = response.content
HuggingFace_output = hf_response

emb_model = OpenAIEmbeddings()

OpenAI_ref_vec = emb_model.embed_documents([reference])[0]
OpenAI_out_vec = emb_model.embed_documents([OpenAI_output])[0]

OpenAI_similarity = cosine_similarity([OpenAI_ref_vec], [OpenAI_out_vec])[0][0]
print(f"OpenAI semantic similarity with reference: {OpenAI_similarity:.2f}")


# HuggingFace RAG output
HuggingFace_output = hf_response

HuggingFace_ref_vec = emb_model.embed_documents([reference])[0]
HuggingFace_out_vec = emb_model.embed_documents([HuggingFace_output])[0]

HuggingFace_similarity = cosine_similarity([HuggingFace_ref_vec], [HuggingFace_out_vec])[0][0]
print(f"HuggingFace semantic similarity with reference: {HuggingFace_similarity:.2f}")


OpenAI semantic similarity with reference: 0.92
HuggingFace semantic similarity with reference: 0.86


##Conclusion

In [ ]:
from IPython.display import Markdown, display

markdown_text = """
# 🔍 **Conclusion: Semantic Similarity Comparison**

Based on the semantic similarity results using both **OpenAI** and **HuggingFace** embeddings, the following conclusions can be drawn:

---

## 🟦 **OpenAI Embeddings — Higher Semantic Alignment**

OpenAI embeddings demonstrated **higher semantic similarity**, indicating that the generated RAG response is more closely aligned with the expected reference answer.

**OpenAI is preferable when:**
- ✅ **High accuracy** and deeper semantic understanding are required
- ✅ **Cloud inference** is acceptable
- ✅ **Cost is not a major constraint**
- ✅ **Faster inference** is preferred
- ✅ You need consistent, high-quality embeddings across diverse domains

---

## 🟩 **HuggingFace Embeddings — Strong, Cost-Effective Alternative**

HuggingFace embeddings delivered **competitive performance** while offering significant practical advantages:

**HuggingFace is ideal when:**
- 💰 **Zero cost** is important
- 🔐 **Local execution** (offline capability) is required
- 🔒 **Data privacy** or sensitive environments are involved
- 📊 You need scalable embeddings for **large batch evaluations**

---

## 📝 **Summary**

| Criterion | OpenAI | HuggingFace |
|----------|--------|-------------|
| Semantic Accuracy | ⭐ Higher | Good |
| Cost | ❌ Paid | ✅ Free |
| Speed | ⭐ Fast | Moderate (CPU-friendly) |
| Cloud Dependency | Yes | No (local) |
| Best Use Case | High-precision applications | Offline / cost-sensitive workflows |

---

## 🏁 **Final Note**

Both embedding methods performed well, but the optimal choice depends on your project constraints:

- **Choose OpenAI → when accuracy is the top priority**
- **Choose HuggingFace → when cost and privacy matter most**
"""

display(Markdown(markdown_text))



# 🔍 **Conclusion: Semantic Similarity Comparison**

Based on the semantic similarity results using both **OpenAI** and **HuggingFace** embeddings, the following conclusions can be drawn:

---

## 🟦 **OpenAI Embeddings — Higher Semantic Alignment**

OpenAI embeddings demonstrated **higher semantic similarity**, indicating that the generated RAG response is more closely aligned with the expected reference answer.

**OpenAI is preferable when:**
- ✅ **High accuracy** and deeper semantic understanding are required
- ✅ **Cloud inference** is acceptable
- ✅ **Cost is not a major constraint**
- ✅ **Faster inference** is preferred
- ✅ You need consistent, high-quality embeddings across diverse domains

---

## 🟩 **HuggingFace Embeddings — Strong, Cost-Effective Alternative**

HuggingFace embeddings delivered **competitive performance** while offering significant practical advantages:

**HuggingFace is ideal when:**
- 💰 **Zero cost** is important
- 🔐 **Local execution** (offline capability) is required
- 🔒 **Data privacy** or sensitive environments are involved
- 📊 You need scalable embeddings for **large batch evaluations**

---

## 📝 **Summary**

| Criterion | OpenAI | HuggingFace |
|----------|--------|-------------|
| Semantic Accuracy | ⭐ Higher | Good |
| Cost | ❌ Paid | ✅ Free |
| Speed | ⭐ Fast | Moderate (CPU-friendly) |
| Cloud Dependency | Yes | No (local) |
| Best Use Case | High-precision applications | Offline / cost-sensitive workflows |

---

## 🏁 **Final Note**

Both embedding methods performed well, but the optimal choice depends on your project constraints:

- **Choose OpenAI → when accuracy is the top priority**
- **Choose HuggingFace → when cost and privacy matter most**


##References
- AnalyticsVidhya : Training videos and hands on material from AnalyticsVidhya.
- ChatGPT : Python coding support and troubleshooting.